In [ ]:
### 순환 신경망(Recurrent Neural Network)

## IMDB 리뷰 데이터셋 호출 & 추출

from keras.datasets import imdb

(train_input, train_target), (test_input, test_target) = imdb.load_data(num_words = 200)

print("훈련 & 테스트 데이터 세트 크기: ", train_input.shape, test_input.shape)

print("0번째 훈련 샘플 길이(크기): ", len(train_input[0]))
print("1번째 훈련 샘플 길이(크기): ", len(train_input[1]))

print("0번째 훈련 샘플 내용(요소): ", train_input[0]) # 출력 결과 - 어휘 사전(0~199)에 따라 단어 단위 정수 출력
# 어휘 사전에 없는 단어 -> 모두 '2'로 처리

print("맨앞 20개 샘플 타깃(0-부정, 1-긍정): ", train_target[:20])

# 훈련-검증 세트 추출
from sklearn.model_selection import train_test_split
train_input, val_input, train_target, val_target = train_test_split(train_input, train_target, test_size = 0.2, random_state = 42)

import numpy as np

lengths = np.array([len(x) for x in train_input]) # 모든 훈련 샘플 길이를 저장한 리스트
print("훈련 샘플 길이의 평균값 및 중간값: ", np.mean(lengths), np.median(lengths))

# 훈련 샘플 길이의 평균값 및 중간값 시각화
import matplotlib.pyplot as plt
plt.hist(lengths)
plt.xlabel("length")
plt.ylabel("frequency")
plt.show() # 출력 결과 - 길이 0~300 부근에 밀집한 분포

# 훈련 샘플 길이 통일 - pad_sequences 함수 활용

from keras.preprocessing.sequence import pad_sequences

# pad_sequences() 함수 -> 시퀀스 데이터 길이 통일
train_seq = pad_sequences(train_input, maxlen = 100)
print("길이를 맞춘 훈련 샘플 크기: ", train_seq.shape)
print("길이를 맞춘 0번째 훈련 샘플 내용: ", train_seq[0])
print("길이를 맞추기 전 0번째 훈련 샘플 후반부: ", train_input[0][-10:])
# 길이 통일 전/후 0번째 샘플 마지막 10개 요소 동일 -> pad_sequences는 전반부부터 제거.
print("길이를 맞춘 5번째 훈련 샘플 내용: ", train_seq[5])
# 표준 길이보다 길이가 작은 샘플 -> 남은 요소 0으로 패딩

val_seq = pad_sequences(val_input, maxlen = 100)



## 순환 신경망(RNN) 모델 구현

import keras

model = keras.Sequential()
model.add(keras.layers.Input(shape = (100, 200))) # 입력 크기 = (길이, 단어 표현(원-핫 인코딩))
model.add(keras.layers.SimpleRNN(8)) # 단순한 순환 신경망 층 추가
model.add(keras.layers.Dense(1, activation = 'sigmoid')) # 출력층 추가
model.summary()


## 단어 표현(원-핫 인코딩)을 반영한 훈련 샘플 준비

# keras.utils.to_categorical() 함수 -> input에 대한 원-핫 원코딩 배열 반환
train_oh = keras.utils.to_categorical(train_seq)
print(train_oh.shape) # (훈련 샘플 개수 = 20000, 단어 길이 = 100, 단어 표현 = 200)
print(train_oh[0][0][:199])
# 0번째 샘플의 0번째 단어 -> '10' -> 원-핫 인코딩 배열의 9번째(10번) 요소만 '1', 나머지는 '0'

val_oh = keras.utils.to_categorical(val_seq)



## 순환 신경망 훈련

model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

checkpoint_cb = keras.callbacks.ModelCheckpoint('best-simplernn-model.keras', save_best_only = True)

early_stopping_cb = keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)

# 훈련 시 배치 사이즈 및 에포크 횟수 변경(CNN 코드와 비교)
history = model.fit(train_oh, train_target, epochs= 100, batch_size = 64, validation_data=(val_oh, val_target), callbacks = [checkpoint_cb, early_stopping_cb])

# 에포크 당 손실 변화 시각화

import matplotlib.pyplot as plt

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()

# 원-핫 인코딩 단어 표현의 문제점: 상당한 용량 사용
print("원핫 인코딩 적용 전/후 데이터 용량 비교: ", train_seq.nbytes, train_oh.nbytes)
# 원-핫 인코딩 -> 단어(요소)마다 '어휘 사전' 크기에 해당하는 배열 추가 -> 과도한 희소 배열 문제



## 단어 임베딩 - 각 단어를 고정된 크기의 실수 벡터로 변환하는 단어 표현 기법

# 어휘 사전 500개 형태로 훈련/테스트 세트 재추출
(train_input, train_target), (test_input, test_target) = imdb.load_data(num_words = 500)
train_input, val_input, train_target, val_target = train_test_split(train_input, train_target, test_size = 0.2, random_state = 42)
train_seq = pad_sequences(train_input, maxlen = 100)
val_seq = pad_sequences(val_input, maxlen = 100)

# 임베딩 층이 추가된 RNN 모델 구현

model_emb = keras.Sequential()
model_emb.add(keras.layers.Input(shape = (100,))) # 단어 표현은 임베딩 층에서 처리
model_emb.add(keras.layers.Embedding(500, 16)) # 어휘 사전 고유한 토큰(요소) 500개를 입력으로 하는 임베딩 층
model_emb.add(keras.layers.SimpleRNN(8)) # 단순한 순환 신경망 층 추가
model_emb.add(keras.layers.Dense(1, activation = 'sigmoid')) # 출력층 추가
model_emb.summary()

model_emb.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
checkpoint_cb = keras.callbacks.ModelCheckpoint('best-embedding-model.keras', save_best_only = True)
early_stopping_cb = keras.callbacks.EarlyStopping(patience = 3, restore_best_weights = True)
history = model_emb.fit(train_seq, train_target, epochs= 100, batch_size = 64, validation_data=(val_seq, val_target), callbacks = [checkpoint_cb, early_stopping_cb])
# 단어 임베딩 -> 원-핫 인코딩보다 사용 용량 & 가중치 개수도 적으면서도 더 나은 성능을 출력함.

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()
